# Cours ULB PHYS-F-450 (2026): Notebook 1 - Les équilibres du modèle de Charney et Straus


Ce notebook est le second d'une série d'exercices pour le cours [PHYS-F-450](https://www.ulb.be/fr/programme/phys-f450) donné à l'ULB.

Dans ce notebook, vous apprendrez :

1. Comment installer le framework [qgs](https://github.com/Climdyn/qgs) pour la modélisation climatique et météorologique d'ordre réduit
2. Comment exécuter le modèle
3. **Déterminer les points d'équilibre du modèle et leur stabilité**
4. Comment calculer le plus grand exposant de Lyapunov d'un modèle, et le lien avec sa prévisibilité
5. Comment calculer le spectre complet des exposants de Lyapunov

Nous nous intéresserons ici à la tâche 3 ci-dessus : apprendre à calculer les points d'équilibre du modèle, [leur stabilité](https://en.wikipedia.org/wiki/Stability_theory) et leur interprétation.

Nous supposons que vous avez déjà suivi le [Notebook 0](https://github.com/jodemaey/cours-ULB-2026/blob/main/Cours%20PHYS-F-450%20--%20Notebook%200%20-%20Introduction.ipynb) d'introduction, qgs étant déjà installé.

## Configuration du modèle

Nous allons utiliser l'atmosphère quasi-géostrophique QG à deux couches vu au cours, tronquée au nombre d'onde 2 suivant la direction méridionale ($y$), et tronquée au nombre d'onde 1 suivant la direction zonale ($x$), sur un beta plane (plan beta) avec une orographie simple (une montagne et une vallée), ce qui nous donnera le modèle de Charney et Straus:

* Charney, J. G., & Straus, D. M. (1980). Form-drag instability, multiple equilibria and propagating planetary waves in baroclinic, orographically forced, planetary wave systems. Journal of Atmospheric Sciences, 37(6), 1157-1176. https://doi.org/10.1175/1520-0469(1980)037%3C1157:FDIMEA%3E2.0.CO;2

Nous allons calculer ses points d'équilibres et leur stabilité, connectant ainsi à la fin du chapitre 5 du cours.

### Import des modules

Nous devons d'abord importer les modules nécessaires aux calculs:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit  # module numba nécessaire pour accélérer les calculs

et importer les modules requis de qgs:

In [ ]:
from qgs.params.params import QgParams
from qgs.integrators.integrator import RungeKuttaIntegrator, RungeKuttaTglsIntegrator
from qgs.functions.tendencies import create_tendencies
from qgs.plotting.util import std_plot

et les "diagnostics":

In [ ]:
from qgs.diagnostics.streamfunctions import MiddleAtmosphericStreamfunctionDiagnostic
from qgs.diagnostics.temperatures import MiddleAtmosphericTemperatureAnomalyDiagnostic
from qgs.diagnostics.variables import VariablesDiagnostic
from qgs.diagnostics.multi import MultiDiagnostic

### Définition du modèle

Nous allons maintenant créer l'objet des paramètres du modèle. Cet objet définit entièrement la version et les paramètres du modèle. Il sera utilisé par qgs pour construire les tendances $\boldsymbol{f}$ du modèle.

In [ ]:
model_parameters = QgParams({'n': 1.3})  # nous fixons le rapport d'aspect n du domaine du modèle à 1.3 pour le moment

Nous pouvons maintenant configurer certains paramètres avec la méthode [`set_params`](https://qgs.readthedocs.io/en/latest/files/technical/configuration.html#qgs.params.params.Params.set_params) de l'objet [`QgParams`](https://qgs.readthedocs.io/en/latest/files/technical/configuration.html#qgs.params.params.QgParams) :

In [ ]:
# nous définissons ici la latitude à 45 degrés et une amplitude prédéfinie du gradient de température méridien
model_parameters.set_params({'phi0_npi': np.deg2rad(45.)/np.pi, 'hd':0.045, 'kd': 0.1, 'kdp': 0.01, 'sigma':0.2})

et indiquer que nous souhaitons un canal atmosphérique pour l'atmosphère, avec des modes de Fourier jusqu'au nombre d'onde 2 dans la direction méridionale $y$ et jusqu'au nombre d'onde 1 dans la direction zonale $x$ :

In [ ]:
model_parameters.set_atmospheric_channel_fourier_modes(1, 2, mode='symbolic')
# on utilise le mode symbolique pour forcer qgs à calculer les coefficients, et non à utiliser une formule prédéfinie
# c'est une des possibilités dans qgs

Nous définissons également une certaine orographie :

In [ ]:
model_parameters.ground_params.set_orography(0.2, 1)

et indiquons l'amplitude du gradient de la température méridionale d'équilibre qui "force" le modèle :

In [ ]:
model_parameters.atemperature_params.set_thetas(0.1, 0)

le `0` indiquant ici qu'on force suivant le premier mode $\sim \cos(y)$ où $y \in [0, \pi]$ est la "latitude" du modèle (voir plus bas). On a donc une température d'équilibre plus élevée au sud qu'au nord du domaine.

La configuration du modèle est terminée. Nous pouvons maintenant afficher les paramètres pour vérifier notre travail :

In [ ]:
model_parameters.print_params()

Nous sommes maintenant prêts à fournir l'objet contenant les paramètres à qgs pour obtenir les tendances $\boldsymbol{f}$ qui nous permettront d'intégrer l'équation différentielle $\dot{\boldsymbol{x}} = \boldsymbol{f}(\boldsymbol{x})$. Pour pouvoir faire la même analyse que dans le cours, nous allons aussi demander les coefficients des projections des équations du modèle (les $a_i$ et $c_{ijk}$ dans le cours). Ils vont être stockés dans l'objet `inner_products` (voir la documentation de la fonction [`create_tendencies`](https://qgs.readthedocs.io/en/latest/files/technical/functions.html#qgs.functions.tendencies.create_tendencies).

In [ ]:
%%time
f, Df, inner_products = create_tendencies(model_parameters, return_inner_products=True)

Trois objets sont en réalité retournés, les tendances $\boldsymbol{f}$ mais aussi son Jacobien $\boldsymbol{Df}$, qui sera utile plus tard pour calculer les exposants de Lyapunov, et finalement les produits intérieurs (les coefficients des projections des équations du modèle).

On peut déjà confirmer en regardant les coefficients $c_{ijk}$ ce qui est avancé au cours sur les termes non-linéaires:

In [ ]:
# cette ligne montre les coefficients cachés dans la structure du code
inner_products[0]._g.todense()

à comparer avec par exemple la valeur $i=1, j=2, k=3$ de $-\frac{8\sqrt{2}}{3\pi} n$  (ici $n=1.3$): 

In [ ]:
print(-8* np.sqrt(2) * model_parameters.scale_params.n / (3 * np.pi))
print(inner_products[0].g(0,1,2)) # en indice python, 1,2,3 devient 0,1,2

Ces coefficients ont été calculés directement par le code.

## Recherche des équilibres

### Recherche des racines

Maintenant que nous avons la fonction $\boldsymbol{f}$, nous pouvons trouver les équilibres en résolvant l'équation $\boldsymbol{f} = 0$:

In [ ]:
# Par souci de commodité, nous définissons une fonction sans dépendance temporelle ; 
# de toute façon, notre fonction f ne dépend pas du temps.
@njit
def fnt(x):
    return f(0.,x)

# importons une fonction pour résoudre l'équation ci-dessus
from scipy.optimize import root

eps = 1.e-6
fixed_point = list()

# on va tirer 2000 conditions initiales pour l'algorithme de recherche des racines
for i in range(2000):
    ic = np.random.rand(model_parameters.ndim)
    sol = root(fnt, ic)
    if sol.success:
        # si on a trouvé une solution, on teste qu'on ne l'a pas déjà trouvée
        for x in fixed_point:
            if np.linalg.norm(x-sol.x) < eps:
                break
        else:
            fixed_point.append(sol.x)
            

# conversion du résultat final en array numpy
fixed_point = np.array(fixed_point)

Voici la liste des points fixes trouvés:

In [ ]:
fixed_point

### Dessin des champs correspondants

Avant de nous intéresser à leur stabilité, essayons de dessiner les champs atmosphériques qu'ils représentent: nous allons présenter pour chaque équilibre le champ de hauteur géopotentielle à 500 hPa relié à la fonction de courant barotrope, ainsi que la température, reliée à la fonction de courant barocline par la relation du vent thermique. Pour cela nous allons nous aider des diagnostics du modèle qgs. Pour plus de détails sur les diagnostics et leur utilisation pour produire ce type d'animation, veuillez consulter cette [section](https://qgs.readthedocs.io/en/latest/files/user_guide.html#analyzing-the-model-s-output-with-diagnostics) du guide de l'utilisateur.

In [ ]:

psi = MiddleAtmosphericStreamfunctionDiagnostic(model_parameters, geopotential=True, delta_x=0.2, delta_y=0.2)
theta = MiddleAtmosphericTemperatureAnomalyDiagnostic(model_parameters, delta_x=0.2, delta_y=0.2)

m = MultiDiagnostic(1,2)
m.add_diagnostic(psi,
                 diagnostic_kwargs={'style': 'contour', 'contour_labels': False, 'show_time': False},
                 plot_kwargs={'colors': 'k'})
m.add_diagnostic(theta, diagnostic_kwargs={'show_time':False})

m.set_data(np.arange(1., fixed_point.shape[0]+1), fixed_point.T)

m.animate(figsize=(20,6))

En principe, en bougeant le slider "time_index", vous pouvez plotter séquentiellement chaque point d'équilibre. Comme dans le papier original, certains équilibres correspondent à des "[ridges](https://en.wikipedia.org/wiki/Ridge_(meteorology))" ou à des "[troughs](https://en.wikipedia.org/wiki/Trough_(meteorology))" centrés sur la "vallée" du modèle (en bleu).  Un des points est un mode purement zonal appelé la solution de Hadley dans l'article du modèle et dans le cours. Pour ce point, seulement les coefficients $\psi_1$ et $\theta_1$ correspondant au mode en $\cos(y)$ sont non-nuls. Il y a aussi un équilibre presque zonal appelé la solution "near" Hadley dans le papier.

Nous pouvons maintenant nous tourner vers le calcul de la stabilité de ces points.

### Stabilité des équilibres trouvés

Grâce à la matrice Jacobienne $\boldsymbol{Df}(\boldsymbol{x})$ d'un équilibre $\boldsymbol{x}$, on peut déterminer sa stabilité. Il suffit de calculer les valeurs propres de cette matrice. Par exemple, pour la solution de Hadley:

In [ ]:
# trouvons d'abord la solution de Hadley
for i in range(fixed_point.shape[0]):
    not_hadley = False
    for j in range(fixed_point.shape[1]):
        if abs(fixed_point[i][j]) > eps and j not in [0, 6]:
            not_hadley = True
    if not not_hadley:
        break
else:
    raise RuntimeError("La solution de Hadley n'a pas été trouvée parmi les équilibres. "
                       "Relancez la recherche des racines (cellule précédente), "
                       "éventuellement avec plus de conditions initiales.")


# ensuite évaluons la matrice Jacobienne
J = Df(0., fixed_point[i])

On peut noter que la structure de cette matrice correspond à celle donnée au cours:

In [ ]:
J

Quant à ses valeurs propres, on obtient:

In [ ]:
np.linalg.eigvals(J)

avec certaines des parties réelles des valeurs propres positives, indiquant que la solution de Hadley est instable pour les paramètres sélectionnés. Ceci est cohérent avec la figure 21 du cours.

On peut aussi calculer la stabilité de tous les équilibres:

In [ ]:
for i in range(fixed_point.shape[0]):
    eigvals = np.linalg.eigvals(Df(0., fixed_point[i]))
    print(eigvals)
    for val in eigvals:
        if np.real(val) > 0.:
            print(f"\033[1mL'équilibre {i} est instable !\033[0m")
            break
    else:
        print(f"\033[1mL'équilibre {i} est stable !\033[0m")
    

Ici, tous les équilibres semblent instables, et l'on peut maintenant passer à la détermination des trajectoires du modèle (qui ne convergeront donc pas en principe vers un point fixe, un équilibre).

### Intégration temporelle (trajectoires du modèle)

D'abord fixons quelques paramètres d'intégration

In [ ]:
# Pas de temps d'intégration
dt = 0.1
# On veut sauver l'état du modèle tous les 5 pas de temps
write_steps = 5
# Durée pendant laquelle on intègre un transient initial 
# du modèle (en unité de temps du modèle)
transient_time = 200000.
# Temps d'intégration des trajectoires du modèle
integration_time = 300.

Nous pouvons maintenant intégrer notre modèle à l'aide d'un intégrateur, un algorithme qui nous fournira la solution de l'équation différentielle ci-dessus. Il existe de nombreuses options pour ce faire (voir plus bas), mais nous utiliserons ici l'intégrateur [`RungeKuttaIntegrator`](https://qgs.readthedocs.io/en/latest/files/technical/integrators.html#qgs.integrators.integrator.RungeKuttaIntegrator) intégré à qgs (un intégrateur de [Runge-Kutta d'ordre 4](https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods)) :

In [ ]:
integrator = RungeKuttaIntegrator()

Nous demanderons à cet intégrateur d'utiliser notre modèle nouvellement défini:

In [ ]:
integrator.set_func(f)

Nous pouvons maintenant partir d'une petite condition initiale aléatoire et intégrer sur une période transitoire pour obtenir une condition initiale sur les attracteurs.

In [ ]:
%%time
ic = np.random.rand(model_parameters.ndim)*0.1
integrator.integrate(0., transient_time, dt, ic=ic, write_steps=write_steps)  # avec write_steps=0, l'intégrateur gardera seulement le point final en mémoire
time, transient = integrator.get_trajectories()

(voir les options et arguments de la fonction [`integrate`](https://qgs.readthedocs.io/en/latest/files/technical/integrators.html#qgs.integrators.integrator.RungeKuttaIntegrator.integrate) de l'intégrateur, qui lui demande ici d'effectuer l'intégration temporelle).

Il n'est pas inintéressant de tracer l'évolution temporelle cette première trajectoire transitoire, par exemple celle de la première variable: 

In [ ]:
var = 0
plt.figure(figsize=(10, 8))

plt.plot(model_parameters.dimensional_time*time, transient[var])

plt.xlabel('time (days)')
plt.ylabel(r'$\psi_1$');

et l'on voit qu'en fait le modèle converge assez rapidement vers son attracteur, dû au caractère *dissipatif* du modèle. Noter aussi qu'on a converti les unités temporelles du modèle en jours via un facteur de conversion. On peut donc utiliser le dernier point de la trajectoire comme condition initiale pour obtenir une trajectoire entièrement sur celui-ci:

In [ ]:
ic = transient[:, -1]

Maintenant que nous avons obtenu une condition initiale sur l'attracteur, nous pouvons donc intégrer davantage pour obtenir une trajectoire sur l'attracteur.

In [ ]:
%%time
integrator.integrate(0., integration_time, dt, ic=ic, write_steps=write_steps)
time, trajectory = integrator.get_trajectories()

### Tracé de la trajectoire du modèle

L'évolution temporelle typique d'une variable du modèle en fonction du temps peut maintenant être représentée graphiquement, par exemple pour la deuxième variable :

In [ ]:
var = 1
plt.figure(figsize=(10, 8))

plt.plot(model_parameters.dimensional_time*time, trajectory[var])

plt.xlabel('time (days)')
plt.ylabel(r'$\psi_2$');

et l'on voit que la solution du modèle est en fait une orbite périodique.

On peut également représenter graphiquement, par exemple, les deuxième et troisième variables dans un plan.

In [ ]:
varx = 2
vary = 1
plt.figure(figsize=(10, 8))

plt.plot(trajectory[varx], trajectory[vary], marker='o', ms=0.07, ls='-')

plt.xlabel(r'$\psi_3$')
plt.ylabel(r'$\psi_2$');

### Représentation graphique des champs spatiaux du modèle

Nous pouvons calculer le champ spatial de la hauteur géopotentielle à 500 hPa pour cette solution et le faire évoluer le long de l'orbite périodique comme dans le précédent notebook.

In [ ]:
psi = MiddleAtmosphericStreamfunctionDiagnostic(model_parameters, geopotential=True, delta_y=0.2, delta_x=0.2)
variable_nondim = VariablesDiagnostic([2, 1], model_parameters, False)
background = VariablesDiagnostic([2, 1], model_parameters, False)
background.set_data(time, trajectory)

m = MultiDiagnostic(1,2)
m.add_diagnostic(variable_nondim,
                 diagnostic_kwargs={'show_time': False, 'background': background},
                 plot_kwargs={'ms': 0.2})
m.add_diagnostic(psi,
                 diagnostic_kwargs={'style': 'contour', 'contour_labels': False},
                 plot_kwargs={'colors': 'k'})

m.set_data(time, trajectory)

m.animate(figsize=(20,6))

Comment interprétez vous ce résultat ?

## Exercices

A faire par exemple en continuant ce notebook:

* Un premier exercice simple est de redémarrer ce notebook pour voir si du chaos apparaît dans le modèle de Charney et Straus comme dans le cours  en changeant:
    - le paramètre $n$ et en lui donnant la valeur $1.77$
    - la valeur de $\theta^\ast_1$ en $0.18$ (initialement ce paramètre vaut $0.1$)
    - la valeur de $\sigma$ en $0.4$ (initialement ce paramètre vaut $0.2$)
    - la valeur de $h_d$ ($h''$ dans le cours), $k_d$ et $k'_d$ en $0.0114$.
    - Il faudra sans doute augmenter le temps d'intégration `integration_time` pour bien le voir.
* Un autre exercice est d'essayer de reproduire alors la figure 21 du cours.